In [0]:
import java.time.LocalDate
import org.apache.spark.sql.{SparkSession, Row}
import org.apache.spark.sql.types._
import scala.util.Try
import scala.collection.JavaConverters._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.DateType
import java.time.format.DateTimeFormatter
import org.apache.spark.sql.{DataFrame, Row}
import org.apache.spark.sql.types.{StructType, StructField, StringType}
import org.apache.spark.sql.Column

In [0]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder().appName("Conexion_Desconexion_Router").getOrCreate()

var parsing_in_base = dbutils.widgets.get("parsing_in_base_json")
var parsing_out = dbutils.widgets.get("parsing_out_csv")

val parsing_in_wildcard = s"$parsing_in_base/*/*/*/*/*/*"

// Función para eliminar archivos y subcarpetas dentro de una carpeta específica
def deleteCurrentFolderOnly(path: String): Unit = {
  try {
    val filesAndDirs = dbutils.fs.ls(path)
    if (filesAndDirs.nonEmpty) {
      filesAndDirs.foreach(file => dbutils.fs.rm(file.path, true))
    }
    dbutils.fs.rm(path, true)
    println(s"Se eliminó correctamente la carpeta y su contenido: $path")
  } catch {
    case e: Exception => println(s"[ERROR] No se pudo eliminar la carpeta $path: ${e.getMessage}")
  }
}

try {
  // Leer los archivos como texto
  val jsonLines = spark.read.text(parsing_in_wildcard).as[String]

  // Convertir cada línea en un objeto JSON
  val df = spark.read.json(jsonLines)

  // Aplanar los campos anidados
  val flattenedDf = df.select(
    col("EnqueuedTimeUtc"),
    col("Properties.hubName").as("hubName"),
    col("Properties.deviceId").as("deviceId"),
    col("Properties.opType").as("opType"),
    col("Properties.iothub-message-schema").as("iothub-message-schema"),
    col("Properties.operationTimestamp").as("operationTimestamp"),
    col("SystemProperties.correlationId").as("correlationId"),
    col("SystemProperties.connectionDeviceId").as("connectionDeviceId"),
    col("SystemProperties.contentType").as("contentType"),
    col("SystemProperties.contentEncoding").as("contentEncoding"),
    col("SystemProperties.userId").as("userId"),
    col("SystemProperties.enqueuedTime").as("enqueuedTime"),
    col("Body.sequenceNumber").as("sequenceNumber")
  )

  // Guarda el archivo en la carpeta de salida
  flattenedDf.repartition(1).write.option("header", "true").option("delimiter", ";").mode("append").csv(parsing_out)

  // Extraer el nombre de la carpeta
  val pattern = ".*/stage_pre/([^/]+)/\\*.*".r
  val extractedName = parsing_in_wildcard match {
    case pattern(name) =>
      val datePart = name.substring(0, 8)  // "20250331"
      val hourPart = name.substring(8, 10) // "12"
      s"desconexion_conexion_router_${hourPart}_${datePart}.csv"
    case _ => "No encontrado"
  }

  // Variable para almacenar la ruta final del archivo renombrado
  var finalFilePath = ""

  // Renombrar el archivo guardado con el filename construido
  val files = dbutils.fs.ls(parsing_out)
  val oldFilePathOption = files.find(file => file.name.startsWith("part-"))

  oldFilePathOption match {
    case Some(oldFile) =>
      val oldFilePath = oldFile.path
      val newFilePath = parsing_out + extractedName
      dbutils.fs.mv(oldFilePath, newFilePath)
      finalFilePath = newFilePath
      println(s"El archivo ha sido renombrado a: $newFilePath")
    case None =>
      println("No se encontró ningún archivo que cumpla con el criterio.")
  }

  // Llamar a la función para eliminar la carpeta de entrada
  deleteCurrentFolderOnly(parsing_in_base)

  // Registrar la ruta como salida del notebook
  dbutils.notebook.exit(finalFilePath)

} catch {
  case e: Exception =>
    println("[ERROR] " + e)
    throw e
}

println("[INFO] PROCESO TERMINADO")